In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
sys.path.append(os.path.abspath('..'))
import numpy as np
import torch
import torch.nn as nn
# matplotlib.use('Agg')  # Headless plotting
import matplotlib.pyplot as plt
from timeit import default_timer
from darcy.config import *  # expects device, seed
from darcy.parameter import *
from darcy.waveletfamily import *
from darcy.coeff_selection import *
from darcy.awpinns import AWPINN
from tqdm import tqdm
import numpy as np
seed = 121
torch.manual_seed(seed)
np.random.seed(seed)

/envs/common/lib/python3.12/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [2]:
# load data 
data_test = np.load("data/test_dataset_out.npz", allow_pickle=True)
k = torch.tensor(data_test["k"], dtype=torch.float32)
f = torch.tensor(data_test["f"], dtype=torch.float32)
p_exact = torch.tensor(data_test["p"], dtype=torch.float32)
perm_params = data_test["perm_params"]
forcing_params = data_test["forcing_params"]

dataset = np.load("data/darcy_pred_fno_out.npz", allow_pickle=True)
p_fno_pred = torch.tensor(dataset["predictions"], device=device)

In [4]:
dx = (x_fno[-1] - x_fno[0])/(n_fno-1)
dy = (y_fno[-1] - y_fno[0])/(n_fno-1)
Jx = torch.arange(0, 6)
Jy = torch.arange(0, 6)
a = 0.3
family = wavelet_family(Jx, Jy, a)
jx, jy, kx, ky = family[:,0], family[:,1], family[:,2], family[:,3]

Err =[]
def train_awpinn_for_fno(i):
    chunk_size = 128
    p_vec = p_fno_pred[i].reshape(-1)
    # ===================== COMPUTE COEFF =====================
    b = apply_WT(p_vec, x_fno.to(device), y_fno.to(device), jx, jy, kx, ky, chunk_size=chunk_size) * dx * dy
    coeff = conjugate_gradient(b, dx, dy, x_fno.to(device), y_fno.to(device), jx, jy, kx, ky, max_iter=1, tol=1e-2)

    # ===================== RECONSTRUCTION =====================
    u_recon_vec = apply_W(coeff, x_fno.to(device), y_fno.to(device), jx, jy, kx, ky, chunk_size=chunk_size)
    u_recon = u_recon_vec.reshape(p_fno_pred.shape[-2], p_fno_pred.shape[-1])
    bias = (u_recon_vec - p_vec).mean()
    top_indices = torch.nonzero(abs(coeff) > torch.quantile(abs(coeff), 0.7), as_tuple=True)[0]

    selected_indices = torch.unique(top_indices)

    coeff_new = coeff[selected_indices]
    family_new = family[selected_indices]

    print("coeff_len: ", len(coeff_new))
    print("Family len: ", len(family_new))

    wx = family_new[:,0].float().to(device)
    bx = family_new[:,2].float().to(device)
    wy = family_new[:,1].float().to(device)
    by = family_new[:,3].float().to(device)

    AWPINN_model = AWPINN(wx, bx, wy, by, coeff_new, bias).to(device)

    optimizer_AWPINN = torch.optim.LBFGS(AWPINN_model.parameters(), 
                                lr=1.0,                            
                                max_iter=5000,
                                max_eval=10**10,
                                tolerance_grad=1e-10,
                                tolerance_change=1e-10,
                                history_size=50,
                                line_search_fn=None)
    x_interior = x_collocation.clone()
    y_interior = y_collocation.clone()
    k_p, dk_dx_p, dk_dy_p = permeability_derivatives(x_interior, y_interior, perm_params[i], device=device)
    rhs = forcing_function(x_interior, y_interior, forcing_params[i])
    u_bc_left = torch.zeros_like(y_bc)
    u_bc_right = torch.zeros_like(y_bc)
    u_bc_bottom = torch.zeros_like(x_bc)
    u_bc_top = torch.zeros_like(x_bc)
    exact = p_exact[i] 
    def awpinn_loss():   

        u, u_x, u_y, u_xx, u_yy = AWPINN_model(x_interior, y_interior)
        # print(u.shape, u_x.shape, u_y.shape, u_xx.shape, u_yy.shape, k_p1.shape, kx_p1.shape, ky_p1.shape, rhs.shape)

        pde_loss = (torch.mean((- (k_p * (u_yy + u_xx) + dk_dx_p * u_x + dk_dy_p * u_y) - rhs)**2))
        
        u_pred_bc_left, _, _, _, _ = AWPINN_model(x_bc_left, y_bc)
        u_pred_bc_right, _, _, _, _ = AWPINN_model(x_bc_right, y_bc)
        u_pred_bc_bottom, _, _, _, _ = AWPINN_model(x_bc, y_bc_bottom)
        u_pred_bc_top, _, _, _, _ = AWPINN_model(x_bc, y_bc_top)

        
        bc_loss = torch.mean((u_pred_bc_left - u_bc_left) ** 2) +\
                torch.mean((u_pred_bc_right - u_bc_right) ** 2) +\
                torch.mean((u_pred_bc_bottom - u_bc_bottom) ** 2) + \
                torch.mean((u_pred_bc_top - u_bc_top) ** 2)
        lam_bc = 500 
        total_loss = pde_loss + lam_bc * bc_loss
        return total_loss, pde_loss, bc_loss

    def train_awpinn(optimizer):

        global itr
        itr = 0

        def closure():
            global itr
            optimizer.zero_grad()
            total_loss, pde_loss, bc_loss = awpinn_loss()

            total_loss.backward()

            if itr % 1000 == 0 or itr == 100:
                with torch.no_grad():

                    numerical, _, _, _, _ = AWPINN_model(x_test.to(device), y_test.to(device))
                
                    errL2 = (torch.sum(torch.abs(exact.reshape(-1).to(device)-numerical)**2))**0.5 / (torch.sum(torch.abs(exact.reshape(-1).to(device))**2))**0.5
                    errMax = torch.max(torch.abs(exact.reshape(-1).to(device)-numerical))

                print(f'Epoch[{itr}]  '
                        f'Total Loss: {total_loss.item():.6f}, '
                        f'PDE Loss: {pde_loss.item():.6f}, '
                        f'BC Loss: {bc_loss.item():.6f}\n\t\t'
                        f'RelativeL2: {errL2},\t'
                        f'Max: {errMax}\n' )
                
                torch.cuda.empty_cache()

                
            itr += 1

            return total_loss

        loss = optimizer.step(closure)

    loss = train_awpinn(optimizer_AWPINN)
    with torch.no_grad():
        numerical, _, _, _, _ = AWPINN_model(x_test.to(device), y_test.to(device))
    return numerical.cpu(), exact.cpu()

In [5]:
for i in tqdm(range(0,10)):
    numerical, exact_test = train_awpinn_for_fno(i)
    ErrL2 = (torch.sum(torch.abs(exact_test.reshape(-1)-numerical.reshape(-1))**2))**0.5 / (torch.sum(torch.abs(exact_test.reshape(-1))**2))**0.5
    ErrMax = torch.max(torch.abs(exact_test.reshape(-1)-numerical.reshape(-1)))
    print(f'Test Relative L2 Error: {ErrL2.item()}, Test Max Error: {ErrMax.item()}')
    Err.append((i, ErrL2.item(), ErrMax.item()))

  0%|          | 0/10 [00:00<?, ?it/s]

coeff_len:  3831
Family len:  3831


/home/subham/HDD/Document/SP_HP_FINAL_AWPINNs/Darcy/darcy/awpinns.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.output_bias = nn.Parameter(torch.tensor(bias))


Epoch[0]  Total Loss: 8.998156, PDE Loss: 8.984190, BC Loss: 0.000028
		RelativeL2: 0.2025061994791031,	Max: 0.010200299322605133

Epoch[100]  Total Loss: 0.091105, PDE Loss: 0.068619, BC Loss: 0.000045
		RelativeL2: 0.15617644786834717,	Max: 0.009255140088498592

Epoch[1000]  Total Loss: 0.002153, PDE Loss: 0.001693, BC Loss: 0.000001
		RelativeL2: 0.020740613341331482,	Max: 0.0013295721728354692

Epoch[2000]  Total Loss: 0.001527, PDE Loss: 0.001449, BC Loss: 0.000000
		RelativeL2: 0.01633959822356701,	Max: 0.0009753052145242691

Epoch[3000]  Total Loss: 0.001357, PDE Loss: 0.001311, BC Loss: 0.000000
		RelativeL2: 0.008240487426519394,	Max: 0.000410346343414858

Epoch[4000]  Total Loss: 0.001180, PDE Loss: 0.001135, BC Loss: 0.000000
		RelativeL2: 0.011204037815332413,	Max: 0.0004325669433455914



 10%|█         | 1/10 [19:05<2:51:47, 1145.29s/it]

...

Test Relative L2 Error: 0.04346621409058571, Test Max Error: 0.0007264914456754923
coeff_len:  3831
Family len:  3831
Epoch[0]  Total Loss: 16.945616, PDE Loss: 16.913874, BC Loss: 0.000063
		RelativeL2: 0.24203599989414215,	Max: 0.02308204025030136

Epoch[100]  Total Loss: 0.181835, PDE Loss: 0.138889, BC Loss: 0.000086
		RelativeL2: 0.11781708896160126,	Max: 0.012463534250855446

Epoch[1000]  Total Loss: 0.003397, PDE Loss: 0.002814, BC Loss: 0.000001
		RelativeL2: 0.011072780936956406,	Max: 0.0030849380418658257

Epoch[2000]  Total Loss: 0.002208, PDE Loss: 0.002028, BC Loss: 0.000000
		RelativeL2: 0.008741261437535286,	Max: 0.001837469986639917

Epoch[3000]  Total Loss: 0.001775, PDE Loss: 0.001645, BC Loss: 0.000000
		RelativeL2: 0.007266148924827576,	Max: 0.0008934751385822892

Epoch[4000]  Total Loss: 0.001354, PDE Loss: 0.001271, BC Loss: 0.000000
		RelativeL2: 0.008737477473914623,	Max: 0.0007548492285422981



 90%|█████████ | 9/10 [2:57:05<19:54, 1194.50s/it]

Test Relative L2 Error: 0.007462019100785255, Test Max Error: 0.00048770138528198004
coeff_len:  3831
Family len:  3831
Epoch[0]  Total Loss: 63.740738, PDE Loss: 63.552525, BC Loss: 0.000376
		RelativeL2: 0.37235233187675476,	Max: 0.04609683156013489

Epoch[100]  Total Loss: 0.490063, PDE Loss: 0.361157, BC Loss: 0.000258
		RelativeL2: 0.35006362199783325,	Max: 0.037740811705589294

Epoch[1000]  Total Loss: 0.014791, PDE Loss: 0.013274, BC Loss: 0.000003
		RelativeL2: 0.03779858723282814,	Max: 0.004133284091949463

Epoch[2000]  Total Loss: 0.006828, PDE Loss: 0.006217, BC Loss: 0.000001
		RelativeL2: 0.028047697618603706,	Max: 0.003402926027774811

Epoch[3000]  Total Loss: 0.005122, PDE Loss: 0.004862, BC Loss: 0.000001
		RelativeL2: 0.026955043897032738,	Max: 0.0034166276454925537

Epoch[4000]  Total Loss: 0.004498, PDE Loss: 0.004343, BC Loss: 0.000000
		RelativeL2: 0.028817052021622658,	Max: 0.003580823540687561



100%|██████████| 10/10 [3:16:09<00:00, 1176.97s/it]

Test Relative L2 Error: 0.028941502794623375, Test Max Error: 0.0035828426480293274


In [ ]:
Err

array([[0.00000000e+00, 1.13789374e-02, 3.65559215e-04],
       [1.00000000e+00, 3.52301225e-02, 1.69380382e-03],
       [2.00000000e+00, 3.69825307e-03, 4.49691201e-04],
       [3.00000000e+00, 4.62451112e-03, 4.10864217e-04],
       [4.00000000e+00, 6.92205429e-02, 5.50165121e-03],
       [5.00000000e+00, 4.76915911e-02, 1.04971393e-03],
       [6.00000000e+00, 3.45551111e-02, 1.06943341e-03],
       [7.00000000e+00, 4.37136851e-02, 7.27260718e-04],
       [8.00000000e+00, 7.77996657e-03, 4.92243969e-04],
       [9.00000000e+00, 2.84606200e-02, 3.54561955e-03]])

In [6]:
np.savez("data/log/AWPINNs_darcy_mean_error_out.npz", Err=Err)